## Symmetric Repair Analyzer

In [ ]:
import pandas as pd

# You can read the already calculated repairs file, or read the one generated on 't-box repairs analyzer' folder with the script 't-box_repairs_analyzer.py'
repairs = pd.read_csv("../../datasets/symmetric_repairs.csv")

repairs

- The cell below counts different types of basic T-box repairs generated with the relational database:

In [ ]:
print(len(repairs[(repairs['C_deleted'] == True)]))
print(len(repairs[(repairs['C_deprecated'] == True)]))
print(len(repairs[(repairs['CQ_added_exception'] == True)]))

- check for base statement deletions:

In [ ]:
repairs['S_deleted'] = False

In [ ]:
import pandas as pd
import requests
import xml.etree.ElementTree as ET
import os
import time
from tqdm import tqdm

def isRemovedWithObj(subject, property, obj, max_attempts=3):
    # URL of the endpoint
    endpoint = "ENTER_qEndpoint_WD_2023"

    # SPARQL query
    query = f"ASK {{ <{subject}> <{property}> <{obj}> }}"

    # URL encode the query
    encoded_query = requests.utils.quote(query)
    
    # Build the complete URL
    url = f"{endpoint}?query={encoded_query}"
    
    # Send HTTP GET request
    headers = {"Accept": "application/xhtml+xml,application/xml;"}
    
    for attempt in range(1, max_attempts + 1):
        response = requests.get(url, headers=headers)
        
        # Check if the request was successful and parse the response
        if response.ok:
            # Parse the XML response
            root = ET.fromstring(response.text)
            boolean_element = root.find('.//{http://www.w3.org/2005/sparql-results#}boolean')
            if boolean_element is not None:
                return boolean_element.text.lower() == 'false'
            else:
                print("Error: 'boolean' element not found in XML response")
                return None
        else:
            print(f"Error on attempt {attempt} for subject: {subject}, property: {property}, object: {obj}")
        
        if attempt < max_attempts:
            time.sleep(1)  # Optional: wait for 1 second before retrying
    
    return None

def process_repairs(repairs, checkpoint_file='symmetric/checkpoint.csv'):
    # Check if there is a checkpoint file and load it
    if os.path.exists(checkpoint_file):
        processed_repair = pd.read_csv(checkpoint_file)
        start_index = len(processed_repair)
        print(f"Checkpoint found. Resuming from index {start_index}.")
    else:
        processed_repair = pd.DataFrame(columns=repairs.columns.tolist())
        start_index = 0

    # Create a progress bar
    with tqdm(total=len(repairs), initial=start_index, desc="Processing repairs") as pbar:
        for index in range(start_index, len(repairs)):
            row = repairs.iloc[index]
            
            subject = row['subject']
            property = row['property']
            obj = row['object']
            
            if (index % 10000 == 0) and (index > start_index):
                print(f"Processing row {index}. Saving checkpoint.")
                # Save checkpoint
                processed_repair.to_csv(checkpoint_file, index=False)
            
            if (index % 10000 == 0):
                print(f"Processed {index} rows.")
            
            if obj.startswith('http'):
                # Call isRemovedWithObj function
                removed = isRemovedWithObj(subject, property, obj)
                if removed is not None:
                    # Update S_deleted column in repairs DataFrame
                    repairs.at[index, 'S_deleted'] = removed
                    # Create a new row for processed_repair DataFrame
                    new_row = repairs.loc[index].tolist()
                    processed_repair.loc[index] = new_row
                else:
                    print(f"Error in: {subject} {property} {obj}")

            # Update the progress bar
            pbar.update(1)

    # Save the final result
    processed_repair.to_csv(checkpoint_file, index=False)
    print("Processing complete. Final checkpoint saved.")

In [ ]:
# Call the function to process base statement deletion repairs
process_repairs(repairs)

In [ ]:
len(repairs[(repairs['S_deleted'] == True)])

- testing addition of context statement p(o,s):

In [ ]:
repairs['Sc_added'] = None

In [ ]:
import requests
import xml.etree.ElementTree as ET

def symmetricAdded(subject, property, obj, max_attempts=3):
    # URL of the endpoint
    endpoint = "ENTER_qEndpoint_WD_2023"

    # SPARQL query
    query = f"ASK {{ <{obj}> <{property}> <{subject}> }}"

    # URL encode the query
    encoded_query = requests.utils.quote(query)
    
    # Build the complete URL
    url = f"{endpoint}?query={encoded_query}"
    
    # Send HTTP GET request
    headers = {"Accept": "application/xhtml+xml,application/xml;"}
    
    for attempt in range(1, max_attempts + 1):
        response = requests.get(url,headers=headers)

        # Check if the request was successful and parse the response
        if response.ok:
            # Parse the XML response
            #print(response.text)
            root = ET.fromstring(response.text)
            boolean_element = root.find('.//{http://www.w3.org/2005/sparql-results#}boolean')
            if boolean_element is not None:
                return boolean_element.text.lower() == 'true'
            else:
                print("Error: 'boolean' element not found in XML response")
                return None
        else:
            print(f"Error on attempt {attempt} for subject: {subject}, property: {property}, object: {obj}")
        
        if attempt < max_attempts:
            time.sleep(1)  # Optional: wait for 1 second before retrying
        
    return None
# Example usage
subject = "http://www.wikidata.org/entity/Q10925338"
property = "http://www.wikidata.org/prop/direct/P1322"
obj = "http://www.wikidata.org/entity/Q24840677"
print(symmetricAdded(subject, property, obj))

In [ ]:
from tqdm import tqdm

# Wrap the dataframe with tqdm to show progress
for index, row in tqdm(repairs.iterrows(), total=len(repairs), desc="Processing rows"):
    # Extract subject and property without the prefix "http://www.wikidata.org/entity/"
    subject = row['subject']
    property = row['property']
    obj = row['object']
    
    if (index % 100000 == 0):
        print(index)
    #break
    
    if (obj.startswith('http') and row['S_deleted'] == False):
        # Call Sc_added function
        symm = symmetricAdded(subject, property, obj)

        # Update S_deleted column
        repairs.at[index, 'Sc_added'] = symm
    else:
        repairs.at[index, 'Sc_added'] = False

# Display the updated DataFrame
#print(filtered_df)

In [ ]:
len(repairs[(repairs['Sc_added'] == True)])

- checking non-categorized yet repairs:

In [ ]:
repairs[(repairs['Sc_added'] == False) & 
     (repairs['S_deleted'] == False) & 
     (repairs['CQ_added_exception'] == False)& 
     (repairs['C_deprecated'] == False)& 
     (repairs['C_deleted'] == False)
    ]

- checking the deletion of base statement when the object is a blank node:

In [ ]:
import requests
import xml.etree.ElementTree as ET

def isRemoved(subject, property):
    # URL of the endpoint
    endpoint = "ENTER_qEndpoint_WD_2023"

    # SPARQL query
    query = f"ASK {{ <{subject}> <{property}> [] }}"

    # URL encode the query
    encoded_query = requests.utils.quote(query)
    
    # Build the complete URL
    url = f"{endpoint}?query={encoded_query}"
    
    # Send HTTP GET request
    headers = {"Accept": "application/xhtml+xml,application/xml;"}
    
    response = requests.get(url,headers=headers)

    # Check if the request was successful and parse the response
    if response.ok:
        # Parse the XML response
        #print(response.text)
        root = ET.fromstring(response.text)
        boolean_element = root.find('.//{http://www.w3.org/2005/sparql-results#}boolean')
        if boolean_element is not None:
            return boolean_element.text.lower() == 'false'
        else:
            print("Error: 'boolean' element not found in XML response")
            return None
    else:
        # If there's an error in the request, return None
        print("Error:", response.text)
        return None

# Example usage
subject = "http://www.wikidata.org/entity/Q456784"
property = "http://www.wikidata.org/prop/direct/P190"
print(isRemoved(subject, property))

In [ ]:
from tqdm import tqdm

# Wrap the dataframe with tqdm to show progress
for index, row in tqdm(repairs.iterrows(), total=len(repairs), desc="Processing rows"):
    
    if (row['object'].startswith('genid') and row['S_deleted'] == False and row['Sc_added'] == False):
        subject = row['subject']
        property = row['property']
        
        # Call isRemovedWithObj function
        removed = isRemoved(subject, property)

        # Update S_deleted column
        repairs.at[index, 'S_deleted'] = removed

# Display the updated DataFrame
#print(filtered_df)

In [ ]:
repairs[(repairs['Sc_added'] == False) & 
     (repairs['S_deleted'] == False)
    ]

In [ ]:
repairs.to_csv("symmetric_repairs.csv", index=False)

In [ ]:
from tqdm import tqdm

# Wrap the dataframe with tqdm to show progress
for index, row in tqdm(repairs.iterrows(), total=len(repairs), desc="Processing rows"):
    if (row['S_deleted'] == None):
        subject = row['subject']
        property = row['property']
        
        # Call isRemovedWithObj function
        removed = isRemoved(subject, property)

        # Update S_deleted column
        result.at[index, 'S_deleted'] = removed

In [ ]:
len(repairs)

- setting triples with blank node objects to have False context statement addition (the alue was None):

In [ ]:
import numpy as np

# Assuming repairs is your DataFrame

# Filter rows where object starts with 'genid' and Sc_added is None
mask = repairs['object'].str.startswith('genid') & repairs['Sc_added'].isna()

# Set Sc_added to False for the filtered rows
repairs.loc[mask, 'Sc_added'] = False

- recalculating shares of repairs:

In [ ]:
len(repairs[(repairs['C_deleted'] == True)])

In [ ]:
len(repairs[(repairs['C_deprecated'] == True)])

In [ ]:
len(repairs[(repairs['CQ_added_exception'] == True)])

In [ ]:
len(repairs[(repairs['S_deleted'] == True)])

In [ ]:
len(repairs[(repairs['Sc_added'] == True)])

In [ ]:
repairs.to_csv("final_symmetric_repairs.csv", index=False)

- generating Venn diagram with all repair shares:

In [ ]:
pip install matplotlib upsetplot

In [ ]:
pip install matplotlib-venn

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib_venn import venn2

# Create the new DataFrame with the required columns
df2 = pd.DataFrame()
df2['A-box changes'] = repairs['S_deleted'] | repairs['Sc_added']
df2['T-box changes'] = (
    repairs['C_deleted'] | 
    repairs['C_deprecated'] | 
    repairs['CQ_added_exception']
)

# Calculate the sizes of the sets
a_box_changes = df2['A-box changes'].sum()
t_box_changes = df2['T-box changes'].sum()
intersection = (df2['A-box changes'] & df2['T-box changes']).sum()

# Plot the Venn diagram
venn2(subsets=(a_box_changes, t_box_changes, intersection), 
      set_labels=('A-box changes', 'T-box changes'))


# Add title
plt.title("Symmetric Constraint share of repairs")

plt.show()